# Conditional Inference Results

This notebook inspects the two u128 conditional inference runs:

1. **Discrete class-conditional field model**: `nf_class_conditional_u128`, conditioned on field label only.
2. **Continuous conditional cosmology model**: `nf_conditional_u128`, conditioned on six CAMELS parameters.

It is designed to run from the repo root on Great Lakes after sampling jobs finish. Missing files are reported rather than treated as hard failures. The sample labels are explicit so the notebook does not accidentally switch between raw DDPM-style samples and any DPM/DDPM comparison files in the same directory.


In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from simdiff_eval.metrics import batch_power_spectra

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'scripts').exists()), Path.cwd()).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

SEED = int(os.environ.get('CONDITIONAL_SEED', 123))
MAX_IMAGES_PER_GROUP = int(os.environ.get('CONDITIONAL_MAX_IMAGES_PER_GROUP', 6))
CLASS_REAL_MAX_PER_FIELD = int(os.environ.get('CLASS_REAL_MAX_PER_FIELD', 512))
CLASS_PK_NBINS = int(os.environ.get('CLASS_PK_NBINS', 25))
CLASS_SWEEP = 'nf_class_conditional_u128'
CONT_SWEEP = 'nf_conditional_u128'
CLASS_SAMPLE_LABEL = os.environ.get('CLASS_SAMPLE_LABEL', 'raw_class_conditional')
CONT_SAMPLE_LABEL = os.environ.get('CONT_SAMPLE_LABEL', 'raw_conditional')
OUT_DIR = PROJECT_DIR / 'results' / 'conditional_inference'
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'font.size': 11,
})

print('project:', PROJECT_DIR)
print('seed:', SEED)
print('output:', OUT_DIR)

## Utilities

In [ ]:
def read_json(path: Path) -> Any:
    with path.open() as f:
        return json.load(f)


def manifest_row(sweep: str) -> dict[str, Any] | None:
    path = PROJECT_DIR / 'local' / sweep / 'manifest.json'
    if not path.exists():
        print('missing manifest:', path)
        return None
    rows = read_json(path)
    if not rows:
        print('empty manifest:', path)
        return None
    if isinstance(rows, dict):
        return rows
    if len(rows) > 1:
        print(f'{path}: expected one row, found {len(rows)}; using first row')
    return rows[0]


def resolve_project_path(raw: str | Path | None) -> Path | None:
    if raw is None:
        return None
    path = Path(str(raw))
    if path.is_absolute():
        if path.exists():
            return path
        # Some cached local manifests contain absolute paths from a different clone.
        marker = 'diffusion_models_repo'
        parts = path.parts
        if marker in parts:
            suffix = Path(*parts[parts.index(marker) + 1:])
            return PROJECT_DIR / suffix
        marker = 'Diffusion_model'
        if marker in parts:
            suffix = Path(*parts[parts.index(marker) + 1:])
            return PROJECT_DIR / suffix
        return path
    return PROJECT_DIR / path


def _format_candidate(raw: str | Path, row: dict[str, Any]) -> Path:
    return resolve_project_path(str(raw).format(seed=SEED, run_name=row.get('run_name', '')))


def resolve_sample_path(row: dict[str, Any], sweep: str, suffix_hint: str, sample_label: str | None = None) -> Path | None:
    run = row.get('run_name', '*')
    sample_root = PROJECT_DIR / 'results' / sweep / 'samples'
    candidates: list[Path] = []

    raw = row.get('sample_path')
    if raw:
        candidates.append(_format_candidate(raw, row))

    if sample_label:
        candidates.append(sample_root / f'{run}_seed{SEED}_{sample_label}.npz')
        candidates.extend(sorted(sample_root.glob(f'{run}_seed{SEED}_*{sample_label}*.npz')))

    candidates.extend(sorted(sample_root.glob(f'{run}_seed{SEED}_*.npz')))
    candidates.extend(sorted(sample_root.glob(f'*seed{SEED}*{suffix_hint}*.npz')))

    seen: set[Path] = set()
    unique_candidates = []
    for path in candidates:
        if path is None or path in seen:
            continue
        seen.add(path)
        unique_candidates.append(path)

    if sample_label:
        for path in unique_candidates:
            if path.exists() and sample_label in path.name:
                return path
    for path in unique_candidates:
        if path.exists():
            return path
    return unique_candidates[0] if unique_candidates else None


def load_npz_samples(path: Path) -> np.ndarray:
    with np.load(path) as data:
        key = 'samples' if 'samples' in data.files else data.files[0]
        arr = np.asarray(data[key], dtype=np.float32)
    if arr.ndim == 3:
        arr = arr[:, None, :, :]
    if arr.ndim != 4 or arr.shape[1] != 1:
        raise ValueError(f'Expected (N,1,H,W) or (N,H,W), got {arr.shape} from {path}')
    return arr


def sample_stats(images: np.ndarray, group: np.ndarray | None = None, group_name: str = 'group') -> pd.DataFrame:
    flat = images.reshape(len(images), -1)
    df = pd.DataFrame({
        'sample_mean': flat.mean(axis=1),
        'sample_std': flat.std(axis=1),
        'sample_p01': np.percentile(flat, 1, axis=1),
        'sample_p50': np.percentile(flat, 50, axis=1),
        'sample_p99': np.percentile(flat, 99, axis=1),
    })
    if group is not None:
        df[group_name] = group[:len(df)]
    return df


def finite_summary(images: np.ndarray) -> dict[str, float | int | tuple[int, ...]]:
    arr = np.asarray(images)
    finite = np.isfinite(arr)
    return {
        'shape': tuple(arr.shape),
        'finite': int(finite.sum()),
        'total': int(arr.size),
        'min': float(np.nanmin(arr)),
        'max': float(np.nanmax(arr)),
        'mean': float(np.nanmean(arr)),
        'std': float(np.nanstd(arr)),
    }


def robust_limits(*arrays: np.ndarray) -> tuple[float, float]:
    vals = np.concatenate([np.asarray(a).ravel() for a in arrays if np.asarray(a).size])
    lo, hi = np.nanpercentile(vals, [1, 99])
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = -1.0, 1.0
    return float(lo), float(hi)


def savefig(fig, name: str) -> Path:
    path = OUT_DIR / name
    fig.savefig(path, bbox_inches='tight')
    print('saved', path)
    return path


def tanh_transform_array(images: np.ndarray, alpha: float, beta: float, gamma: float, delta: float, sigma: float, mu: float = 0.0) -> np.ndarray:
    shifted = images - np.float32(mu)
    pos = alpha * np.tanh((gamma * shifted) / alpha)
    neg = beta * np.tanh((delta * shifted) / beta)
    return (np.where(shifted >= 0, pos, neg) * sigma).astype(np.float32, copy=False)


def load_yaml(path: Path) -> dict[str, Any]:
    import yaml
    with path.open() as f:
        return yaml.safe_load(f)


def class_config(row: dict[str, Any]) -> dict[str, Any]:
    config_path = resolve_project_path(row.get('config'))
    if config_path is None or not config_path.exists():
        raise FileNotFoundError(f'Missing class config: {config_path}')
    return load_yaml(config_path)


def class_fields(row: dict[str, Any]) -> list[str]:
    return list(row.get('fields') or [id_to_field[i] for i in sorted(id_to_field)])


def class_data_paths(row: dict[str, Any]) -> dict[str, Path]:
    fields = class_fields(row)
    paths = row.get('data_paths') or []
    return {field: Path(paths[i]) for i, field in enumerate(fields) if i < len(paths)}


def _read_zthinned_slices(path: Path, sim_indices: np.ndarray, zthin: int) -> np.ndarray:
    arr = np.load(path, mmap_mode='r')
    sims = np.asarray(arr[sim_indices, ::zthin], dtype=np.float32)
    return sims.reshape(-1, sims.shape[-2], sims.shape[-1])


def _safe_log_images(images: np.ndarray) -> np.ndarray:
    return np.log(np.maximum(images.astype(np.float32, copy=False), np.float32(1.0e-12)))


def compute_or_load_class_norm_stats(row: dict[str, Any], config: dict[str, Any]) -> dict[str, float | str | int]:
    cache_path = OUT_DIR / 'nf_class_conditional_u128_real_normalization.json'
    fields = class_fields(row)
    n_train = int(row.get('n_train_simulations_per_field', 0) or 0)
    zthin = int(row.get('zthin', config['data'].get('zthin', 1)))
    data_paths = class_data_paths(row)
    key = {
        'run_name': row.get('run_name'),
        'fields': fields,
        'n_train_simulations_per_field': n_train,
        'zthin': zthin,
        'transform': config['data'].get('transform'),
        'normalization': config['data'].get('normalization'),
    }
    if cache_path.exists():
        cached = read_json(cache_path)
        if cached.get('key') == key:
            return cached['stats']

    data_cfg = config['data']
    transform = data_cfg.get('transform', None)
    use_log = bool(data_cfg.get('log', False)) or (isinstance(transform, (list, tuple)) and 'log' in transform)
    norm_kwargs = dict(data_cfg.get('norm_kwargs') or {})
    normalization = data_cfg.get('normalization')

    center = norm_kwargs.get('center', None)
    xmax = norm_kwargs.get('xmax', None)
    chunk = int(os.environ.get('CLASS_NORM_SIM_CHUNK', 32))

    if center is None:
        print('computing real-field normalization center from full class training set...', flush=True)
        total = 0.0
        count = 0
        for field in fields:
            print(f'  center pass: {field}', flush=True)
            path = data_paths[field]
            if not path.exists():
                raise FileNotFoundError(path)
            for start in range(0, n_train, chunk):
                stop = min(start + chunk, n_train)
                images = _read_zthinned_slices(path, np.arange(start, stop), zthin)
                if use_log:
                    images = _safe_log_images(images)
                total += float(images.sum(dtype=np.float64))
                count += int(images.size)
        center = total / max(count, 1)

    if xmax is None:
        print('computing real-field normalization xmax from full class training set...', flush=True)
        xmax_val = 0.0
        for field in fields:
            print(f'  xmax pass: {field}', flush=True)
            path = data_paths[field]
            for start in range(0, n_train, chunk):
                stop = min(start + chunk, n_train)
                images = _read_zthinned_slices(path, np.arange(start, stop), zthin)
                if use_log:
                    images = _safe_log_images(images)
                xmax_val = max(xmax_val, float(np.max(np.abs(images - np.float32(center)))))
        xmax = max(xmax_val, 1e-30)

    stats = {
        'normalization': str(normalization),
        'use_log': bool(use_log),
        'center': float(center),
        'xmax': float(xmax),
        'n_train_simulations_per_field': int(n_train),
        'zthin': int(zthin),
    }
    cache_path.write_text(json.dumps({'key': key, 'stats': stats}, indent=2) + '\n')
    print('wrote', cache_path)
    return stats


def normalize_class_real_images(images: np.ndarray, config: dict[str, Any], norm_stats: dict[str, Any]) -> np.ndarray:
    data_cfg = config['data']
    norm_kwargs = dict(data_cfg.get('norm_kwargs') or {})
    out = images.astype(np.float32, copy=False)
    if norm_stats.get('use_log'):
        out = _safe_log_images(out)
    normalization = data_cfg.get('normalization')
    if normalization in {'tanh', 'centermax', 'center-max', 'centered_maxabs'}:
        out = (out - np.float32(norm_stats['center'])) / np.float32(norm_stats['xmax'])
    if normalization == 'tanh':
        out = tanh_transform_array(
            out,
            alpha=float(norm_kwargs.get('alpha', 1.0)),
            beta=float(norm_kwargs.get('beta', 1.0)),
            gamma=float(norm_kwargs.get('gamma', 1.0)),
            delta=float(norm_kwargs.get('delta', 1.0)),
            sigma=float(norm_kwargs.get('sigma', 1.0)),
            mu=float(norm_kwargs.get('mu', 0.0)),
        )
    elif normalization in {None, 'none', 'centermax', 'center-max', 'centered_maxabs'}:
        pass
    else:
        raise ValueError(f'Unsupported normalization: {normalization!r}')
    return out[:, None, :, :].astype(np.float32, copy=False)


def load_class_real_reference(row: dict[str, Any], max_per_field: int = 512) -> tuple[dict[str, np.ndarray], dict[str, Any]]:
    config = class_config(row)
    norm_stats = compute_or_load_class_norm_stats(row, config)
    fields = class_fields(row)
    data_paths = class_data_paths(row)
    n_train = int(row.get('n_train_simulations_per_field', 0) or 0)
    zthin = int(row.get('zthin', config['data'].get('zthin', 1)))
    slices_per_sim = 128 // zthin
    sims_needed = max(1, int(np.ceil(max_per_field / max(slices_per_sim, 1))))
    sims_needed = min(sims_needed, n_train)
    sim_indices = np.linspace(0, n_train - 1, sims_needed, dtype=np.int64)

    out: dict[str, np.ndarray] = {}
    for field in fields:
        path = data_paths[field]
        if not path.exists():
            raise FileNotFoundError(path)
        raw = _read_zthinned_slices(path, sim_indices, zthin)[:max_per_field]
        out[field] = normalize_class_real_images(raw, config, norm_stats)
    return out, norm_stats


def summarize_by_field(images_by_field: dict[str, np.ndarray], source: str) -> pd.DataFrame:
    rows = []
    for field, images in images_by_field.items():
        stats = sample_stats(images)
        rows.append({
            'source': source,
            'field': field,
            'n': len(images),
            'sample_mean_mean': float(stats['sample_mean'].mean()),
            'sample_mean_std': float(stats['sample_mean'].std()),
            'sample_std_mean': float(stats['sample_std'].mean()),
            'sample_std_std': float(stats['sample_std'].std()),
            'p01_mean': float(stats['sample_p01'].mean()),
            'p50_mean': float(stats['sample_p50'].mean()),
            'p99_mean': float(stats['sample_p99'].mean()),
        })
    return pd.DataFrame(rows)


def histogram_l1(a: np.ndarray, b: np.ndarray, bins: np.ndarray) -> float:
    ha, _ = np.histogram(a.ravel(), bins=bins, density=True)
    hb, _ = np.histogram(b.ravel(), bins=bins, density=True)
    widths = np.diff(bins)
    return float(np.sum(np.abs(ha - hb) * widths))


## Manifest And File Audit

In [ ]:
class_row = manifest_row(CLASS_SWEEP)
cont_row = manifest_row(CONT_SWEEP)

audit_rows = []
for sweep, row, hint, sample_label in [
    (CLASS_SWEEP, class_row, 'class', CLASS_SAMPLE_LABEL),
    (CONT_SWEEP, cont_row, 'conditional', CONT_SAMPLE_LABEL),
]:
    if row is None:
        audit_rows.append({'sweep': sweep, 'status': 'missing manifest'})
        continue
    sample_path = resolve_sample_path(row, sweep, hint, sample_label)
    config_path = resolve_project_path(row.get('config'))
    checkpoint_dir = resolve_project_path(row.get('checkpoint_dir'))
    audit_rows.append({
        'sweep': sweep,
        'run_name': row.get('run_name'),
        'conditioning': row.get('conditioning'),
        'sample_label': sample_label,
        'dataset_size': row.get('dataset_size'),
        'sample_path': str(sample_path) if sample_path else None,
        'sample_exists': bool(sample_path and sample_path.exists()),
        'sample_size_mb': round(sample_path.stat().st_size / 1024**2, 2) if sample_path and sample_path.exists() else np.nan,
        'config_exists': bool(config_path and config_path.exists()),
        'checkpoint_exists': bool(checkpoint_dir and checkpoint_dir.exists()),
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

# Discrete Class-Conditional Field Model

Class IDs are expected to map to CAMELS field types. The default requested mapping is `0=Mcdm`, `1=Mstar`, `2=HI`, `3=Mgas`, `4=Mtot`, `5=ne`.

In [ ]:
if class_row is None:
    class_samples = None
    print('No class-conditional manifest found.')
else:
    class_sample_path = resolve_sample_path(class_row, CLASS_SWEEP, 'class', CLASS_SAMPLE_LABEL)
    class_label_path = resolve_project_path(class_row.get('sample_label_path'))
    class_map_path = resolve_project_path(class_row.get('class_map_path'))
    class_map = class_row.get('class_map') or (read_json(class_map_path) if class_map_path and class_map_path.exists() else {})
    id_to_field = {int(v): str(k) for k, v in class_map.items()}
    print('sample:', class_sample_path)
    print('sample label:', CLASS_SAMPLE_LABEL)
    print('labels:', class_label_path)
    print('class_map:', id_to_field)
    if not class_sample_path or not class_sample_path.exists():
        class_samples = None
        print('Class-conditional sample file missing.')
    elif not class_label_path or not class_label_path.exists():
        class_samples = None
        print('Class-conditional sample label file missing.')
    else:
        class_samples = load_npz_samples(class_sample_path)
        class_labels = np.load(class_label_path).astype(int)
        n = min(len(class_samples), len(class_labels))
        class_samples = class_samples[:n]
        class_labels = class_labels[:n]
        class_names = np.array([id_to_field.get(int(x), f'class_{int(x)}') for x in class_labels])
        print('summary:', finite_summary(class_samples))
        display(pd.Series(class_names).value_counts().rename_axis('field').reset_index(name='n_samples'))

In [ ]:
if 'class_samples' in globals() and class_samples is not None:
    class_stats = sample_stats(class_samples, class_names, 'field')
    class_grouped = class_stats.groupby('field').agg(['mean', 'std', 'min', 'max']).round(5)
    display(class_grouped)
    class_stats_path = OUT_DIR / 'nf_class_conditional_u128_sample_stats.csv'
    class_stats.to_csv(class_stats_path, index=False)
    print('wrote', class_stats_path)
else:
    print('No class samples loaded; skipping class stats.')

## Class-Conditional Real-Reference Checks

These checks compare generated samples against real training/reference fields for each requested class using the same log-floor and tanh normalization as training. This is the main sanity check: each class should match its own field distribution in one-point statistics and P(k), not just produce finite images.


In [ ]:
if 'class_samples' in globals() and class_samples is not None:
    class_real_by_field, class_real_norm = load_class_real_reference(class_row, CLASS_REAL_MAX_PER_FIELD)
    class_gen_by_field = {
        field: class_samples[class_names == field]
        for field in [id_to_field[i] for i in sorted(id_to_field)]
        if np.any(class_names == field)
    }
    print('real reference per field:', {k: v.shape for k, v in class_real_by_field.items()})
    print('real normalization:', class_real_norm)
    real_summary = summarize_by_field(class_real_by_field, 'real_train_ref')
    gen_summary = summarize_by_field(class_gen_by_field, 'generated')
    class_real_gen_summary = pd.concat([real_summary, gen_summary], ignore_index=True)
    display(class_real_gen_summary.round(5))
    out = OUT_DIR / 'nf_class_conditional_u128_real_vs_generated_summary.csv'
    class_real_gen_summary.to_csv(out, index=False)
    print('wrote', out)
else:
    print('No class samples loaded; skipping real-reference load.')


In [ ]:
if 'class_real_by_field' in globals() and class_real_by_field:
    fields = [id_to_field[i] for i in sorted(id_to_field)]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7.8), squeeze=False)
    pdf_rows = []
    for ax, field in zip(axes.ravel(), fields):
        real = class_real_by_field[field]
        gen = class_gen_by_field[field]
        lo, hi = robust_limits(real, gen)
        bins = np.linspace(lo, hi, 160)
        ax.hist(real.ravel(), bins=bins, density=True, histtype='step', lw=2.0, label=f'real train ref n={len(real)}')
        ax.hist(gen.ravel(), bins=bins, density=True, histtype='step', lw=2.0, label=f'generated n={len(gen)}')
        ax.set_title(field)
        ax.set_xlabel('normalized field value')
        ax.set_ylabel('density')
        ax.grid(alpha=0.2)
        ax.legend(frameon=False, fontsize=8)
        pdf_rows.append({
            'field': field,
            'hist_l1': histogram_l1(real, gen, bins),
            'real_mean': float(real.mean()),
            'gen_mean': float(gen.mean()),
            'real_std': float(real.std()),
            'gen_std': float(gen.std()),
        })
    fig.suptitle('Class-conditional generated vs real one-point PDFs', y=1.02)
    fig.tight_layout()
    savefig(fig, 'nf_class_conditional_u128_real_vs_generated_one_point.png')
    plt.show()
    class_pdf_summary = pd.DataFrame(pdf_rows)
    display(class_pdf_summary.round(5))
    out = OUT_DIR / 'nf_class_conditional_u128_real_vs_generated_one_point.csv'
    class_pdf_summary.to_csv(out, index=False)
    print('wrote', out)
else:
    print('No real reference loaded; skipping generated-vs-real one-point PDFs.')


In [ ]:
if 'class_real_by_field' in globals() and class_real_by_field:
    fields = [id_to_field[i] for i in sorted(id_to_field)]
    fig, axes = plt.subplots(2, 3, figsize=(15, 7.8), squeeze=False)
    pk_rows = []
    for ax, field in zip(axes.ravel(), fields):
        real = class_real_by_field[field]
        gen = class_gen_by_field[field]
        pk_real, kbins = batch_power_spectra(real, nbins=CLASS_PK_NBINS)
        pk_gen, _ = batch_power_spectra(gen, nbins=CLASS_PK_NBINS)
        real_mean = np.nanmean(pk_real, axis=0)
        gen_mean = np.nanmean(pk_gen, axis=0)
        ratio = gen_mean / np.clip(real_mean, 1e-30, None)
        ax.plot(kbins, ratio, marker='o', ms=3)
        ax.axhline(1.0, color='k', lw=1, alpha=0.6)
        ax.set_title(field)
        ax.set_xlabel('k bin')
        ax.set_ylabel('generated / real P(k)')
        ax.grid(alpha=0.25)
        pk_rows.append({
            'field': field,
            'pk_log10_mae': float(np.nanmean(np.abs(np.log10(np.clip(ratio, 1e-30, None))))),
            'pk_ratio_mean': float(np.nanmean(ratio)),
            'pk_ratio_min': float(np.nanmin(ratio)),
            'pk_ratio_max': float(np.nanmax(ratio)),
        })
    fig.suptitle('Class-conditional generated vs real power-spectrum ratios', y=1.02)
    fig.tight_layout()
    savefig(fig, 'nf_class_conditional_u128_real_vs_generated_pk_ratio.png')
    plt.show()
    class_pk_summary = pd.DataFrame(pk_rows)
    display(class_pk_summary.round(5))
    out = OUT_DIR / 'nf_class_conditional_u128_real_vs_generated_pk.csv'
    class_pk_summary.to_csv(out, index=False)
    print('wrote', out)
else:
    print('No real reference loaded; skipping generated-vs-real P(k).')


In [ ]:
if 'class_real_by_field' in globals() and class_real_by_field:
    fields = [id_to_field[i] for i in sorted(id_to_field)]
    n_show = min(MAX_IMAGES_PER_GROUP, 4)
    fig, axes = plt.subplots(len(fields), 2 * n_show, figsize=(2.0 * 2 * n_show, 2.0 * len(fields)), squeeze=False)
    for row_idx, field in enumerate(fields):
        gen = class_gen_by_field[field]
        real = class_real_by_field[field]
        vmin, vmax = robust_limits(gen, real)
        gen_idx = np.linspace(0, len(gen) - 1, n_show, dtype=int)
        real_idx = np.linspace(0, len(real) - 1, n_show, dtype=int)
        for col in range(n_show):
            ax = axes[row_idx, col]
            ax.imshow(gen[int(gen_idx[col]), 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_title(f'{field} gen', fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
        for col in range(n_show):
            ax = axes[row_idx, n_show + col]
            ax.imshow(real[int(real_idx[col]), 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_title(f'{field} real', fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle('Generated vs real class samples, per-field color scale', y=1.01)
    fig.tight_layout()
    savefig(fig, 'nf_class_conditional_u128_real_vs_generated_image_grid_per_field_scale.png')
    plt.show()
else:
    print('No real reference loaded; skipping generated-vs-real image grid.')


## Class Label Usage Checks

The first label-usage check below uses the finished sample set: class labels should create much larger between-class shifts than within-class scatter in generated summaries. The optional paired-noise probe below is stricter: it holds the initial noise and reverse-step noise stream fixed and changes only the class label. It is off by default because it runs inference from the notebook.


In [ ]:
if 'class_stats' in globals() and class_stats is not None:
    label_usage_rows = []
    for feature in ['sample_mean', 'sample_std', 'sample_p01', 'sample_p50', 'sample_p99']:
        grouped = class_stats.groupby('field')[feature]
        between = float(grouped.mean().std())
        within = float(grouped.std().mean())
        label_usage_rows.append({
            'feature': feature,
            'between_class_std_of_means': between,
            'mean_within_class_std': within,
            'between_over_within': between / max(within, 1e-12),
        })
    class_label_usage = pd.DataFrame(label_usage_rows)
    display(class_label_usage.round(4))
    out = OUT_DIR / 'nf_class_conditional_u128_label_usage_summary.csv'
    class_label_usage.to_csv(out, index=False)
    print('wrote', out)
else:
    print('No class stats loaded; skipping label-usage summary.')


In [ ]:
RUN_CLASS_LABEL_SWAP = os.environ.get('RUN_CLASS_LABEL_SWAP', '0') == '1'
CLASS_LABEL_SWAP_N_PROBES = int(os.environ.get('CLASS_LABEL_SWAP_N_PROBES', '1'))

if not RUN_CLASS_LABEL_SWAP:
    print('Skipping paired same-noise label-swap probe. Set RUN_CLASS_LABEL_SWAP=1 before starting Jupyter to run it.')
elif 'class_samples' not in globals() or class_samples is None:
    print('No class samples/config loaded; skipping paired label-swap probe.')
else:
    import torch
    import diffusers
    from diffusers import UNet2DModel

    config = class_config(class_row)
    checkpoint_root = resolve_project_path(class_row.get('checkpoint_dir'))
    ckpts = sorted(Path(checkpoint_root).glob('checkpoint-epoch-*'), key=lambda p: int(p.name.rsplit('-', 1)[-1]))
    if not ckpts:
        raise FileNotFoundError(f'No checkpoints under {checkpoint_root}')
    checkpoint = ckpts[-1]
    scheduler_cfg = config['noise_scheduler']
    scheduler_cls = getattr(diffusers, scheduler_cfg['class'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = UNet2DModel.from_pretrained(str(checkpoint)).to(device).eval()

    base_gen = torch.Generator(device=device).manual_seed(SEED)
    base_noise = torch.randn((CLASS_LABEL_SWAP_N_PROBES, 1, 128, 128), device=device, generator=base_gen)
    fields = [id_to_field[i] for i in sorted(id_to_field)]
    label_swap_images = []
    with torch.no_grad():
        for class_id, field in enumerate(fields):
            scheduler = scheduler_cls(**scheduler_cfg.get('kwargs', {}))
            scheduler.set_timesteps(int(scheduler.config.num_train_timesteps))
            images = base_noise.clone()
            step_gen = torch.Generator(device=device).manual_seed(SEED + 1000)
            labels = torch.full((CLASS_LABEL_SWAP_N_PROBES,), class_id, device=device, dtype=torch.long)
            for t in scheduler.timesteps:
                timesteps = torch.full((CLASS_LABEL_SWAP_N_PROBES,), int(t), device=device, dtype=torch.long)
                pred = model(images, timesteps, class_labels=labels, return_dict=False)[0]
                images = scheduler.step(pred, t, images, generator=step_gen).prev_sample
            label_swap_images.append(images.detach().cpu().numpy())
    label_swap_images = np.concatenate(label_swap_images, axis=0)

    fig, axes = plt.subplots(len(fields), CLASS_LABEL_SWAP_N_PROBES, figsize=(2.1 * CLASS_LABEL_SWAP_N_PROBES, 2.1 * len(fields)), squeeze=False)
    for row_idx, field in enumerate(fields):
        row_imgs = label_swap_images[row_idx * CLASS_LABEL_SWAP_N_PROBES:(row_idx + 1) * CLASS_LABEL_SWAP_N_PROBES]
        vmin, vmax = robust_limits(row_imgs)
        for col in range(CLASS_LABEL_SWAP_N_PROBES):
            ax = axes[row_idx, col]
            ax.imshow(row_imgs[col, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_title(f'{field}, noise {col}', fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle('Same initial/reverse noise, changed class label only', y=1.01)
    fig.tight_layout()
    savefig(fig, 'nf_class_conditional_u128_same_noise_label_swap.png')
    plt.show()


In [ ]:
if 'class_samples' in globals() and class_samples is not None:
    fields = [id_to_field[i] for i in sorted(id_to_field)]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    bins = np.linspace(*robust_limits(class_samples), 180)
    for field in fields:
        arr = class_samples[class_names == field]
        if len(arr) == 0:
            continue
        axes[0].hist(arr.ravel(), bins=bins, density=True, histtype='step', lw=1.8, label=field)
    axes[0].set_title('class-conditional pixel distributions')
    axes[0].set_xlabel('normalized field value')
    axes[0].set_ylabel('density')
    axes[0].legend(frameon=False, fontsize=9)

    order = [field for field in fields if field in set(class_stats['field'])]
    data = [class_stats.loc[class_stats.field == field, 'sample_std'] for field in order]
    axes[1].boxplot(data, labels=order, showfliers=False)
    axes[1].set_title('per-sample std by requested class')
    axes[1].set_ylabel('sample std')
    axes[1].tick_params(axis='x', rotation=35)
    fig.tight_layout()
    savefig(fig, 'nf_class_conditional_u128_histograms.png')
    plt.show()
else:
    print('No class samples loaded; skipping class histograms.')

In [ ]:
if 'class_samples' in globals() and class_samples is not None:
    fields = [id_to_field[i] for i in sorted(id_to_field)]
    n_cols = min(MAX_IMAGES_PER_GROUP, 6)
    fig, axes = plt.subplots(len(fields), n_cols, figsize=(2.1 * n_cols, 2.05 * len(fields)), squeeze=False)
    vmin, vmax = robust_limits(class_samples)
    for row_idx, field in enumerate(fields):
        idxs = np.where(class_names == field)[0]
        chosen = idxs[np.linspace(0, len(idxs) - 1, n_cols, dtype=int)] if len(idxs) else []
        for col_idx in range(n_cols):
            ax = axes[row_idx, col_idx]
            if col_idx < len(chosen):
                i = int(chosen[col_idx])
                ax.imshow(class_samples[i, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
                ax.set_title(f'{field} #{i}', fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle('Discrete class-conditional generated samples', y=1.01)
    fig.tight_layout()
    savefig(fig, 'nf_class_conditional_u128_image_grid.png')
    plt.show()
else:
    print('No class samples loaded; skipping class image grid.')

# Continuous Conditional Cosmology Model

This model conditions HI generation on the six CAMELS parameters: `Omega_m`, `sigma_8`, `A_SN1`, `A_AGN1`, `A_SN2`, and `A_AGN2`.

In [ ]:
if cont_row is None:
    cont_samples = None
    print('No continuous-conditional manifest found.')
else:
    cont_sample_path = resolve_sample_path(cont_row, CONT_SWEEP, 'conditional', CONT_SAMPLE_LABEL)
    cont_label_path = resolve_project_path(cont_row.get('sample_label_path'))
    cont_raw_path = resolve_project_path(cont_row.get('sample_raw_params_path'))
    stats_path = resolve_project_path(cont_row.get('param_stats_path'))
    param_names = list(cont_row.get('param_names') or [])
    print('sample:', cont_sample_path)
    print('sample label:', CONT_SAMPLE_LABEL)
    print('normalized labels:', cont_label_path)
    print('raw labels:', cont_raw_path)
    print('stats:', stats_path)
    if not cont_sample_path or not cont_sample_path.exists():
        cont_samples = None
        print('Continuous conditional sample file missing.')
    elif not cont_label_path or not cont_label_path.exists():
        cont_samples = None
        print('Continuous conditional label file missing.')
    else:
        cont_samples = load_npz_samples(cont_sample_path)
        cont_labels_norm = np.load(cont_label_path).astype(np.float32)
        if cont_raw_path and cont_raw_path.exists():
            cont_labels_raw = np.load(cont_raw_path).astype(np.float32)
        elif stats_path and stats_path.exists():
            stats = read_json(stats_path)
            mean = np.asarray(stats['mean'], dtype=np.float32)
            std = np.asarray(stats['std'], dtype=np.float32)
            cont_labels_raw = cont_labels_norm * std + mean
            if not param_names:
                param_names = list(stats.get('param_names', []))
        else:
            cont_labels_raw = cont_labels_norm.copy()
        if not param_names:
            param_names = [f'param_{i}' for i in range(cont_labels_norm.shape[1])]
        n = min(len(cont_samples), len(cont_labels_norm), len(cont_labels_raw))
        cont_samples = cont_samples[:n]
        cont_labels_norm = cont_labels_norm[:n]
        cont_labels_raw = cont_labels_raw[:n]
        print('summary:', finite_summary(cont_samples))
        param_df = pd.DataFrame(cont_labels_raw, columns=param_names)
        display(param_df.describe().T.round(5))

In [ ]:
if 'cont_samples' in globals() and cont_samples is not None:
    cont_stats = sample_stats(cont_samples)
    param_df = pd.DataFrame(cont_labels_raw, columns=param_names)
    cont_stats = pd.concat([cont_stats, param_df], axis=1)
    display(cont_stats[['sample_mean', 'sample_std', 'sample_p01', 'sample_p99', *param_names]].head())
    corr_cols = ['sample_mean', 'sample_std', 'sample_p01', 'sample_p99']
    corr = cont_stats[corr_cols + param_names].corr().loc[corr_cols, param_names]
    display(corr.round(3))
    cont_stats_path = OUT_DIR / 'nf_conditional_u128_sample_stats.csv'
    cont_stats.to_csv(cont_stats_path, index=False)
    print('wrote', cont_stats_path)
else:
    print('No continuous samples loaded; skipping continuous stats.')

In [ ]:
if 'cont_samples' in globals() and cont_samples is not None:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.2), squeeze=False)
    for ax, pname in zip(axes.ravel(), param_names):
        sc = ax.scatter(cont_stats[pname], cont_stats['sample_std'], c=cont_stats['sample_mean'], s=18, cmap='viridis')
        ax.set_xlabel(pname)
        ax.set_ylabel('generated sample std')
        ax.grid(alpha=0.25)
    fig.colorbar(sc, ax=axes.ravel().tolist(), label='generated sample mean', shrink=0.85)
    fig.suptitle('Continuous conditional samples: generated summary vs conditioning parameters', y=1.02)
    savefig(fig, 'nf_conditional_u128_param_scatter.png')
    plt.show()
else:
    print('No continuous samples loaded; skipping parameter scatter plot.')

## Continuous Conditioning Smoothness Checks

For the continuous model, plausible images are not enough. These diagnostics ask whether generated one-point summaries and broad P(k) bands vary smoothly with the requested CAMELS parameters. This is still a first-pass check; a stronger validation would compare against held-out simulations at matched parameters.


In [ ]:
if 'cont_samples' in globals() and cont_samples is not None:
    cont_pk, cont_kbins = batch_power_spectra(cont_samples, nbins=CLASS_PK_NBINS)
    finite_bins = np.where(np.isfinite(np.nanmean(cont_pk, axis=0)))[0]
    bands = np.array_split(finite_bins, 3) if len(finite_bins) else []
    band_names = ['low_k', 'mid_k', 'high_k']
    for band_name, idx in zip(band_names, bands):
        cont_stats[f'log10_pk_{band_name}'] = np.log10(np.nanmean(np.clip(cont_pk[:, idx], 1e-30, None), axis=1))

    response_cols = ['sample_mean', 'sample_std', 'sample_p01', 'sample_p99'] + [f'log10_pk_{name}' for name in band_names if f'log10_pk_{name}' in cont_stats]
    cont_condition_corr = cont_stats[response_cols + param_names].corr().loc[response_cols, param_names]
    display(cont_condition_corr.round(3))
    out = OUT_DIR / 'nf_conditional_u128_condition_response_correlations.csv'
    cont_condition_corr.to_csv(out)
    print('wrote', out)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8.2), squeeze=False)
    ycol = 'log10_pk_mid_k' if 'log10_pk_mid_k' in cont_stats else 'sample_std'
    for ax, pname in zip(axes.ravel(), param_names):
        sc = ax.scatter(cont_stats[pname], cont_stats[ycol], c=cont_stats['sample_std'], s=18, cmap='viridis')
        ax.set_xlabel(pname)
        ax.set_ylabel(ycol)
        ax.grid(alpha=0.25)
    fig.colorbar(sc, ax=axes.ravel().tolist(), label='generated sample std', shrink=0.85)
    fig.suptitle(f'Continuous conditional response: {ycol} vs requested parameters', y=1.02)
    savefig(fig, 'nf_conditional_u128_pk_param_trends.png')
    plt.show()
else:
    print('No continuous samples loaded; skipping continuous P(k)/parameter trend checks.')


In [ ]:
if 'cont_samples' in globals() and cont_samples is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    bins = np.linspace(*robust_limits(cont_samples), 180)
    axes[0].hist(cont_samples.ravel(), bins=bins, density=True, histtype='step', lw=2, color='tab:blue')
    axes[0].set_title('continuous conditional pixel distribution')
    axes[0].set_xlabel('normalized field value')
    axes[0].set_ylabel('density')

    axes[1].hist(cont_stats['sample_mean'], bins=50, alpha=0.65, label='sample mean')
    axes[1].hist(cont_stats['sample_std'], bins=50, alpha=0.65, label='sample std')
    axes[1].set_title('per-sample summaries')
    axes[1].legend(frameon=False)
    fig.tight_layout()
    savefig(fig, 'nf_conditional_u128_histograms.png')
    plt.show()
else:
    print('No continuous samples loaded; skipping continuous histograms.')

In [ ]:
if 'cont_samples' in globals() and cont_samples is not None:
    n_params = len(param_names)
    fig, axes = plt.subplots(n_params, 2, figsize=(5.0, 2.1 * n_params), squeeze=False)
    vmin, vmax = robust_limits(cont_samples)
    for row_idx, pname in enumerate(param_names):
        vals = cont_stats[pname].to_numpy()
        for col_idx, which in enumerate(['min', 'max']):
            idx = int(np.argmin(vals) if which == 'min' else np.argmax(vals))
            ax = axes[row_idx, col_idx]
            ax.imshow(cont_samples[idx, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
            ax.set_title(f'{pname} {which}\nidx={idx}, value={vals[idx]:.3g}', fontsize=9)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle('Continuous conditional generated samples at parameter extremes', y=1.01)
    fig.tight_layout()
    savefig(fig, 'nf_conditional_u128_parameter_extremes.png')
    plt.show()
else:
    print('No continuous samples loaded; skipping parameter-extreme image grid.')

## Great Lakes Sampling Status Commands

If either sample is missing in the audit table, check the jobs and logs directly on Great Lakes. The continuous cosmology sampler needs the concrete `UNet2DConditionModel` checkpoint-load patch, so pull the latest branch before resubmitting.

```bash
cd /home/jiamingp/diffusion_models_repo
git pull --ff-only
sacct -j 51018045,51018033 --format=JobID,JobName%24,State,ExitCode,Elapsed,MaxRSS
ls -lh results/nf_class_conditional_u128/samples/*.npz results/nf_conditional_u128/samples/*.npz
```


## Output Files

In [ ]:
for path in sorted(OUT_DIR.glob('*')):
    if path.is_file():
        print(path, path.stat().st_size)